# scFoundation — Cell-Type Annotation Wrapper (MS dataset)

Faithful re-implementation of the `LinearProbingClassifier` in
`models/scFoundation/model/finetune_model.py` (lines 88-148).

**Freezing pattern (verbatim from the script):**
- `token_emb` frozen, `pos_emb` frozen
- All `encoder` frozen, **then** `encoder.transformer_encoder[-2]` selectively unfrozen
  (only this one transformer layer of the backbone is trainable)
- Head: max-pool over seq dim → `BatchNorm1d(768, affine=False, eps=1e-6)` →
  `Linear(768, 256)` → `ReLU` → `Linear(256, n_class)`
- Input padded to 19264 genes via `OS_scRNA_gene_index.19264.tsv`

**Hyperparameters (NOT pinned in the public scFoundation repo — flagged as deviation):**
  `lr=1e-4, AdamW, weight_decay=0, batch_size=8, epochs=10, seed=0`.

In [1]:
import os, sys, random
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse
from sklearn.metrics import accuracy_score, f1_score

# scFoundation requires its own module path on PYTHONPATH (load.py + finetune_model.py).
SCF_ROOT = "/data/benchmark/models/scFoundation/model"
sys.path.insert(0, SCF_ROOT)
from load import load_model_frommmf, gatherData  # type: ignore
from finetune_model import LinearProbingClassifier  # type: ignore

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
# --- Gene padding to 19264 (scFoundation's fixed input dimension) ---
GENE_INDEX_TSV = Path(SCF_ROOT) / "OS_scRNA_gene_index.19264.tsv"
gene_list_df = pd.read_csv(GENE_INDEX_TSV, header=0, delimiter="\t")
TARGET_GENES = gene_list_df["gene_name"].tolist()
print(f"target gene panel: {len(TARGET_GENES)} genes")

def main_gene_selection(X_df: pd.DataFrame, gene_list: list[str]) -> pd.DataFrame:
    # Same logic as `get_embedding.py::main_gene_selection`: pad missing genes with zeros,
    # then reindex.
    missing = list(set(gene_list) - set(X_df.columns))
    pad = pd.DataFrame(np.zeros((X_df.shape[0], len(missing))), columns=missing, index=X_df.index)
    X_df = pd.concat([X_df, pad], axis=1)
    return X_df[gene_list]

LABEL_COL = "Factor Value[inferred cell type - authors labels]"
train_ad = sc.read("/data/benchmark/data/cellPLM/data/c_data.h5ad")
test_ad = sc.read("/data/benchmark/data/cellPLM/data/filtered_ms_adata.h5ad")
train_ad.obs["celltype"] = train_ad.obs[LABEL_COL].astype(str)
test_ad.obs["celltype"] = test_ad.obs[LABEL_COL].astype(str)
test_obs_names = test_ad.obs_names.tolist()

def to_df(a):
    # scFoundation expects gene names as columns. `c_data` / `filtered_ms_adata` are
    # keyed by ensembl IDs but carry a `gene_name` var column.
    if "gene_name" in a.var.columns:
        cols = a.var["gene_name"].astype(str).tolist()
    else:
        cols = a.var_names.tolist()
    X = a.X.toarray() if issparse(a.X) else a.X
    return pd.DataFrame(X, index=a.obs_names, columns=cols)

train_df = main_gene_selection(to_df(train_ad), TARGET_GENES)
test_df = main_gene_selection(to_df(test_ad), TARGET_GENES)
print("train:", train_df.shape, "test:", test_df.shape)

target gene panel: 19264 genes


train: (7844, 19264) test: (13468, 19264)


In [3]:
# --- Label encoding ---
celltype_cats = sorted(set(train_ad.obs["celltype"].astype(str).unique())
                      | set(test_ad.obs["celltype"].astype(str).unique()))
label_to_int = {c: i for i, c in enumerate(celltype_cats)}
int_to_label = {i: c for c, i in label_to_int.items()}
y_train = np.array([label_to_int[c] for c in train_ad.obs["celltype"].astype(str)])
y_test = np.array([label_to_int[c] for c in test_ad.obs["celltype"].astype(str)])
n_class = len(celltype_cats)
print(f"n_class = {n_class}")

n_class = 18


In [4]:
# --- Build LinearProbingClassifier with n_class output, override the head ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT = Path(SCF_ROOT) / "models" / "models.ckpt"

clf = LinearProbingClassifier(ckpt_path=str(CKPT), frozenmore=True)
clf.build()
# The script's __init__ hard-codes n_class=10; replace fc1 with the correct n_class.
hidden_dim = clf.model_config["encoder"]["hidden_dim"]  # 768
clf.fc1 = nn.Sequential(
    nn.Linear(hidden_dim, 256),
    nn.ReLU(),
    nn.Linear(256, n_class),
)
clf = clf.to(device)

# Confirm the freezing pattern matches the script (token_emb frozen, pos_emb frozen,
# encoder frozen except transformer_encoder[-2]).
trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
print(f"trainable params: {len(trainable)} groups; first 3: {trainable[:3]}")

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

self.pos_emb and self.token_emb also frozen
self.encoder.transformer_encoder  self_attn.in_proj_weight  have grad
self.encoder.transformer_encoder  self_attn.in_proj_bias  have grad
self.encoder.transformer_encoder  self_attn.out_proj.weight  have grad
self.encoder.transformer_encoder  self_attn.out_proj.bias  have grad
self.encoder.transformer_encoder  linear1.weight  have grad
self.encoder.transformer_encoder  linear1.bias  have grad
self.encoder.transformer_encoder  linear2.weight  have grad
self.encoder.transformer_encoder  linear2.bias  have grad
self.encoder.transformer_encoder  norm1.weight  have grad
self.encoder.transformer_encoder  norm1.bias  have grad
self.encoder.transformer_encoder  norm2.weight  have grad
self.encoder.transformer_encoder  norm2.bias  have grad
trainable params: 16 groups; first 3: ['encoder.transformer_encoder.10.self_attn.in_proj_weight', 'encoder.transformer_encoder.10.self_attn.in_proj_bias', 'encoder.transformer_encoder.10.self_attn.out_proj.weight']

In [5]:
# --- DataLoaders (scFoundation forward expects sample_list dict with key 'x') ---
X_train_t = torch.from_numpy(train_df.values.astype(np.float32))
X_test_t = torch.from_numpy(test_df.values.astype(np.float32))
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()

BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-4

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

In [6]:
# --- Train ---
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, clf.parameters()), lr=LR, weight_decay=0)
ce = nn.CrossEntropyLoss()

for epoch in range(1, EPOCHS + 1):
    clf.train()
    total_loss, total_n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = clf({"x": xb, "targets": yb})
        loss = ce(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        total_n += xb.size(0)
    print(f"epoch {epoch:2d}  train_loss={total_loss/total_n:.4f}")

epoch  1  train_loss=0.8602


epoch  2  train_loss=0.5420


epoch  3  train_loss=0.4791


epoch  4  train_loss=0.4313


epoch  5  train_loss=0.4165


epoch  6  train_loss=0.3883


epoch  7  train_loss=0.3749


epoch  8  train_loss=0.3542


epoch  9  train_loss=0.3368


epoch 10  train_loss=0.3325


In [7]:
# --- Predict on test ---
clf.eval()
preds = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = clf({"x": xb, "targets": yb.to(device)})
        preds.append(logits.argmax(1).cpu().numpy())
pred_idx = np.concatenate(preds)
pred_strs = np.array([int_to_label[int(i)] for i in pred_idx])
assert len(pred_strs) == len(test_obs_names)

out = ad.AnnData(np.zeros((len(test_obs_names), 1), dtype=np.float32))
out.obs_names = test_obs_names
out.obs["celltype"] = test_ad.obs["celltype"].astype(str).values
out.obs["predictions"] = pd.Categorical(pred_strs)

OUT_PATH = "/data/benchmark/ct_annotation/scFoundation-annotation-wrapper/ms_annotation.h5ad"
out.write_h5ad(OUT_PATH)

acc = accuracy_score(out.obs["celltype"], out.obs["predictions"])
macro_f1 = f1_score(out.obs["celltype"], out.obs["predictions"], average="macro")
print(f"scFoundation  accuracy={acc:.3f}  macro-F1={macro_f1:.3f}  wrote {OUT_PATH}")

scFoundation  accuracy=0.834  macro-F1=0.702  wrote /data/benchmark/ct_annotation/scFoundation-annotation-wrapper/ms_annotation.h5ad
